In [0]:
# Check current version and row count
from pyspark.sql import functions as F

df_current = spark.read.table("nyc.default.gold_trips_by_zone")
print(f"Current row count: {df_current.count()}")

# Check Delta history
spark.sql("DESCRIBE HISTORY nyc.default.gold_trips_by_zone").show(10, truncate=False)

In [0]:
# Cmd 2 — Simulate a bad write (overwrite with garbage data)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType

bad_schema = StructType([
    StructField("pu_location_id", IntegerType()),
    StructField("pickup_zone", StringType()),
    StructField("pickup_borough", StringType()),
    StructField("total_trips", LongType()),
    StructField("avg_fare", DoubleType()),
    StructField("avg_distance_miles", DoubleType()),
    StructField("avg_duration_minutes", DoubleType()),
    StructField("total_revenue", DoubleType()),
])

bad_data = [(999, "CORRUPTED ZONE", "BAD BOROUGH", 0, 0.0, 0.0, 0.0, 0.0)]

df_bad = spark.createDataFrame(bad_data, schema=bad_schema)

(
    df_bad
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc.default.gold_trips_by_zone")
)

print("Bad write simulated.")
print(f"Row count after bad write: {spark.read.table('nyc.default.gold_trips_by_zone').count()}")

In [0]:
# Confirm the damage
display(spark.read.table("nyc.default.gold_trips_by_zone"))

In [0]:
# Use time travel to read the previous good version
# Version 0 = first write from Gold notebook
# Version 1 = the bad write we just did
# So we read version 0

df_good_version = spark.read.format("delta").option("versionAsOf", 0).table("nyc.default.gold_trips_by_zone")
print(f"Row count at version 0: {df_good_version.count()}")
display(df_good_version.limit(5))

In [0]:
# Restore the table to version 0
spark.sql("RESTORE TABLE nyc.default.gold_trips_by_zone TO VERSION AS OF 0")

print("Table restored.")
print(f"Row count after restore: {spark.read.table('nyc.default.gold_trips_by_zone').count()}")

In [0]:
# Check history again — RESTORE adds a new version entry
spark.sql("DESCRIBE HISTORY nyc.default.gold_trips_by_zone").show(10, truncate=False)